In [11]:
import kaggle
import pandas as pd
import zipfile
import numpy as np
import pyodbc
import sqlalchemy as sa
from datetime import datetime
import random

In [39]:
!kaggle datasets download -d ankitbansal06/retail-orders -f orders.csv --unzip

Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
License(s): CC0-1.0


In [61]:
with zipfile.ZipFile("orders.csv", 'r') as zip_ref:
    zip_ref.extractall("unzipped_orders")

In [63]:
df = pd.read_csv("unzipped_orders/orders.csv")
df.head()

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [65]:
df.isna().sum()

Order Id            0
Order Date          0
Ship Mode           1
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Category            0
Sub Category        0
Product Id          0
cost price          0
List Price          0
Quantity            0
Discount Percent    0
dtype: int64

In [67]:
df['Ship Mode'].unique()

array(['Second Class', 'Standard Class', 'Not Available', 'unknown',
       'First Class', nan, 'Same Day'], dtype=object)

In [73]:
df[df['Ship Mode'] == "Not Available"]

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
5,6,2022-03-13,Not Available,Consumer,United States,Los Angeles,California,90032,West,Furniture,Furnishings,FUR-FU-10001487,50,50,7,3
8,9,2023-03-23,Not Available,Consumer,United States,Los Angeles,California,90032,West,Office Supplies,Binders,OFF-BI-10003910,20,20,3,2
10,11,2023-03-31,Not Available,Consumer,United States,Los Angeles,California,90032,West,Furniture,Tables,FUR-TA-10001539,1470,1710,9,3
11,12,2023-12-25,Not Available,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,TEC-PH-10002033,750,910,4,3


In [79]:
df[df['Ship Mode'].isna()]

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
118,119,2023-07-19,NaN,Corporate,United States,Bristol,Tennessee,37620,South,Office Supplies,Binders,OFF-BI-10003650,140,160,1,5


In [95]:
df['Ship Mode'].replace(["Not Available", "unknown"],np.nan, inplace=True)

C:\Users\sheth\AppData\Local\Temp\ipykernel_3092\2354188516.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Ship Mode'].replace(["Not Available", "unknown"],np.nan, inplace=True)


In [97]:
df['Ship Mode'].unique()

array(['Second Class', 'Standard Class', nan, 'First Class', 'Same Day'],
      dtype=object)

In [103]:
df.columns = df.columns.str.lower().str.replace(' ','_')

In [105]:
df.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [111]:
df['discount'] = (df['list_price']*df['discount_percent'])/100.0

In [113]:
df['sale_price'] = df['list_price'] - df['discount']

In [115]:
df['profit'] = df['sale_price'] - df['cost_price']

In [117]:
df.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5,1.0,19.0,-1.0


In [119]:
df.dtypes

order_id              int64
order_date           object
ship_mode            object
segment              object
country              object
city                 object
state                object
postal_code           int64
region               object
category             object
sub_category         object
product_id           object
cost_price            int64
list_price            int64
quantity              int64
discount_percent      int64
discount            float64
sale_price          float64
profit              float64
dtype: object

In [125]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [127]:
df.dtypes

order_id                     int64
order_date          datetime64[ns]
ship_mode                   object
segment                     object
country                     object
city                        object
state                       object
postal_code                  int64
region                      object
category                    object
sub_category                object
product_id                  object
cost_price                   int64
list_price                   int64
quantity                     int64
discount_percent             int64
discount                   float64
sale_price                 float64
profit                     float64
dtype: object

In [129]:
df.drop(columns=['list_price','discount_percent'], axis=1, inplace=True)

In [135]:
df.drop(columns=['cost_price'], axis=1, inplace=True)

In [137]:
df.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,2,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,5,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,2,1.0,19.0,-1.0


In [143]:
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='object')

In [21]:
engine = sa.create_engine("mssql+pyodbc://SKYWALKER\\SQLEXPRESS/prac_db?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")

In [179]:
df.set_index('order_id', inplace=True)

In [181]:
df.head()

,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
order_id,,,,,,,,,,,,,,,
1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,2,5.2,254.8,14.8
2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,3,21.9,708.1,108.1
3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,2,0.5,9.5,-0.5
4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,5,19.2,940.8,160.8
5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,2,1.0,19.0,-1.0


In [183]:
df.to_sql(name='orders', con = engine, if_exists='replace')

38

In [185]:
### Incremental load
data_incremental = {
    'order_id': [9995, 9996, 9997, 9998, 9999],
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-03', '2024-01-04'],
    'ship_mode': ['Second Class', 'First Class', 'Standard Class', 'Same Day', 'Second Class'],
    'segment': ['Consumer', 'Corporate', 'Home Office', 'Consumer', 'Corporate'],
    'country': ['United States']*5,
    'city': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'],
    'state': ['New York', 'California', 'Illinois', 'Texas', 'Arizona'],
    'postal_code': ['10001', '90001', '60601', '77001', '85001'],
    'region': ['East', 'West', 'Central', 'South', 'West'],
    'category': ['Furniture', 'Office Supplies', 'Technology', 'Furniture', 'Technology'],
    'sub_category': ['Chairs', 'Binders', 'Phones', 'Tables', 'Copiers'],
    'product_id': ['FUR-CH-10001', 'OFF-BI-10002', 'TEC-PH-10003', 'FUR-TA-10004', 'TEC-CO-10005'],
    'quantity': [2, 5, 3, 1, 4],
    'discount': [0.1, 0.2, 0.0, 0.15, 0.05],
    'sale_price': [200.0, 150.0, 300.0, 250.0, 1200.0],
    'profit': [50.0, 30.0, 90.0, 40.0, 300.0]
}

In [187]:
df_incremental = pd.DataFrame(data_incremental)

In [189]:
df_incremental.dtypes

order_id          int64
order_date       object
ship_mode        object
segment          object
country          object
city             object
state            object
postal_code      object
region           object
category         object
sub_category     object
product_id       object
quantity          int64
discount        float64
sale_price      float64
profit          float64
dtype: object

In [191]:
df_incremental['order_date'] = pd.to_datetime(df_incremental['order_date'])

In [193]:
df_incremental.dtypes

order_id                 int64
order_date      datetime64[ns]
ship_mode               object
segment                 object
country                 object
city                    object
state                   object
postal_code             object
region                  object
category                object
sub_category            object
product_id              object
quantity                 int64
discount               float64
sale_price             float64
profit                 float64
dtype: object

In [195]:
query = "SELECT MAX(order_date) AS max_date FROM orders"
max_date_result = pd.read_sql(query, engine)

In [197]:
max_date_result

,max_date
0,2023-12-31


In [199]:
max_date = max_date_result.iloc[0]['max_date']

In [201]:
max_date

Timestamp('2023-12-31 00:00:00')

In [207]:
df_new = df_incremental[df_incremental['order_date'] > max_date]

In [209]:
df_new.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,9995,2024-01-01,Second Class,Consumer,United States,New York,New York,10001,East,Furniture,Chairs,FUR-CH-10001,2,0.10,200.0,50.0
1,9996,2024-01-02,First Class,Corporate,United States,Los Angeles,California,90001,West,Office Supplies,Binders,OFF-BI-10002,5,0.20,150.0,30.0
2,9997,2024-01-03,Standard Class,Home Office,United States,Chicago,Illinois,60601,Central,Technology,Phones,TEC-PH-10003,3,0.00,300.0,90.0
3,9998,2024-01-03,Same Day,Consumer,United States,Houston,Texas,77001,South,Furniture,Tables,FUR-TA-10004,1,0.15,250.0,40.0
4,9999,2024-01-04,Second Class,Corporate,United States,Phoenix,Arizona,85001,West,Technology,Copiers,TEC-CO-10005,4,0.05,1200.0,300.0


In [211]:
df_new.set_index('order_id', inplace=True)

In [213]:
df_new.to_sql(name='orders', con = engine, if_exists='append')

5

In [13]:
### Backfill load
missed_ids = [1000, 2000, 3000]

data_backfill = []
for oid in missed_ids:
    record = {
        'order_id': f"{oid}",
        'order_date': datetime(2023, 12, random.randint(1, 30)),
        'ship_mode': random.choice(['Standard Class', 'Second Class', 'Same Day', 'First Class']),
        'segment': random.choice(['Consumer', 'Corporate', 'Home Office']),
        'country': 'United States',
        'city': random.choice(['New York', 'Chicago', 'San Francisco']),
        'state': random.choice(['New York', 'Illinois', 'California']),
        'postal_code': str(random.randint(10000, 99999)),
        'region': random.choice(['East', 'Central', 'West']),
        'category': random.choice(['Furniture', 'Technology', 'Office Supplies']),
        'sub_category': random.choice(['Chairs', 'Phones', 'Binders']),
        'product_id': f"PID-{oid}",
        'quantity': random.randint(1, 5),
        'discount': round(random.choice([0, 0.1, 0.15, 0.2]), 2),
        'sale_price': round(random.uniform(50, 500), 2),
        'profit': round(random.uniform(10, 100), 2)
    }
    data_backfill.append(record)

In [15]:
df_backfill = pd.DataFrame(data_backfill)

In [17]:
df_backfill.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,1000,2023-12-29,Same Day,Corporate,United States,New York,California,42957,Central,Furniture,Binders,PID-1000,4,0.2,400.64,98.76
1,2000,2023-12-20,First Class,Corporate,United States,New York,Illinois,26190,West,Furniture,Phones,PID-2000,4,0.1,435.03,76.33
2,3000,2023-12-08,Standard Class,Corporate,United States,New York,Illinois,47876,Central,Technology,Phones,PID-3000,5,0.0,155.50,69.08


In [25]:
df_backfill.set_index('order_id', inplace=True)

In [27]:
df_backfill.to_sql(name='orders', con = engine, if_exists='append')

3